In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import librosa
import soundfile
import torch
import torchaudio
from torchaudio import transforms as T
import librosa
import soundfile as sf

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.model_selection import train_test_split
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score

from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

import sys
import os
sys.path.append('./dataset/cats_dogs_dataset')
import utils
import myutils
from myutils import Augmentation

print(torch.cuda.is_available())

#### Data Augmentation

In [ ]:
dataset_path_train = f'dataset/talk_dataset/train/'
dataset_path_test = f'dataset/talk_dataset/test/'

# AUGMENTATION RUN ONCE ONLY #
# augmentator = Augmentation(noise_factor=0.001, time_shift_max=0.2, pitch_step=[2,-2], t_stretch_rate=1.2, volume_scale_factor=[1.5,-1.5])
# for folders in [f for f in os.listdir(dataset_path_train) if os.path.isdir(os.path.join(dataset_path_train, f))]:
#     for item in os.listdir(os.path.join(dataset_path_train, folders)):
#         y, sr = librosa.load(os.path.join(dataset_path_train, folders, item))

#         y_noise = augmentator.noise_inject(y)
#         sf.write(f'{dataset_path_train}{folders}/noise_{item}', y_noise, sr)

#         y_time_shift = augmentator.time_shift(y)
#         sf.write(f'{dataset_path_train}{folders}/ts_{item}', y_time_shift, sr)

#         y_pitch_shift = augmentator.pitch_shift(y, sr)
#         sf.write(f'{dataset_path_train}{folders}/ps_{item}', y_pitch_shift, sr)

#         y_time_stretch = augmentator.time_stretch(y)
#         sf.write(f'{dataset_path_train}{folders}/time_str_{item}', y_time_stretch, sr)

#         y_volume_scale = augmentator.volume_scale(y)
#         sf.write(f'{dataset_path_train}{folders}/vc_{item}', y_volume_scale, sr)
        

#### Feature Extraction

In [ ]:
targets = ['ammar', 'aditya', 'faiszal']

def extract_rows(dataset_path):
    rows = []
    for folder in [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]:
        for item in os.listdir(os.path.join(dataset_path, folder)):
            feat = myutils.feature_extraction(os.path.join(dataset_path, folder, item))
            
            # Flatten MFCC features
            for prefix, key in [("mfcc_mean", "mean_mfcc"), ("mfcc_std", "std_mfcc")]:
                for i, v in enumerate(feat.pop(key), start=1):
                    feat[f"{prefix}_{i}"] = v
            
            # Assign target label
            feat['target'] = next(
                (i for i, t in enumerate(targets) if t in item), None
            )
            rows.append(feat)
    
    return pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)

extracted_train = extract_rows(dataset_path_train)
extracted_test  = extract_rows(dataset_path_test)

In [ ]:
extracted_train.head()

In [ ]:
extracted_test.head()

In [ ]:
# from sklearn.inspection import permutation_importance

# result = permutation_importance(model_svm, X_test, y_test, n_repeats=30, random_state=42)

# # Tampilkan feature importance
# feature_names = extracted_train.drop(columns=['target']).columns.tolist()
# importance_df = pd.DataFrame({
#     'feature'   : feature_names,
#     'importance': result.importances_mean,
#     'std'       : result.importances_std
# }).sort_values('importance', ascending=False)

# print(importance_df.to_string(index=False))

#### Data Preprocess

In [ ]:
scaler = StandardScaler()

X_train = extracted_train.drop(columns=['target'])
y_train = extracted_train['target']
X_test = extracted_test.drop(columns=['target'])
y_test = extracted_test['target']

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#### Training

In [ ]:
model_svm = SVC(kernel='linear')
model_svm.fit(X_train, y_train)

In [ ]:
y_pred = model_svm.predict(X_test) 

print("=" * 40)
print("         EVALUATION RESULTS")
print("=" * 40)
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred, average='macro'):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, average='macro'):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_pred, average='macro'):.4f}")
print("=" * 40)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['ammar', 'aditya', 'faiszal']))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['ammar', 'aditya', 'faiszal'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()